# Step 2 — Extract Text từ TikTok (PaddleOCR 3.x + Colab T4)

**Upload lên Drive trước khi chạy:**
```
MyDrive/PBL7/
    ├── metadata.json
    └── cookies.txt
```
Chạy từng cell theo thứ tự ↓

In [1]:
import subprocess, sys, re
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ── Cập nhật yt-dlp lên bản mới nhất để giảm thiểu lỗi download ──
!pip install -U yt-dlp -q
print('✅ Đã cập nhật yt-dlp lên bản mới nhất!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 60.5 MB/s eta 0:00:00
✅ Đã cập nhật yt-dlp lên bản mới nhất!


In [3]:


def run(cmd, label=None):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    tag = label or cmd[:60]
    if result.returncode != 0:
        print(f'STDERR [{tag}]:', result.stderr[-600:])
    else:
        print(f'OK: {tag}')
    return result

# ── Kiểm tra CUDA version ────────────────────────────────────────────────────
r = run('nvcc --version', 'nvcc --version')
cuda_ver = ''
for line in r.stdout.splitlines():
    if 'release' in line:
        m = re.search(r'release (\d+\.\d+)', line)
        if m:
            cuda_ver = m.group(1)
print(f'\n🖥️  CUDA version: {cuda_ver}')

# ── Chọn paddle index URL theo CUDA version ──────────────────────────────────
major = int(cuda_ver.split('.')[0]) if cuda_ver else 12
minor = int(cuda_ver.split('.')[1]) if cuda_ver and '.' in cuda_ver else 0

if major == 11:
    paddle_index = 'https://www.paddlepaddle.org.cn/packages/stable/cu118/'
    cuda_tag = 'cu118'
elif major == 12 and minor <= 3:
    paddle_index = 'https://www.paddlepaddle.org.cn/packages/stable/cu123/'
    cuda_tag = 'cu123'
else:
    paddle_index = 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'
    cuda_tag = 'cu126'

print(f'📦 Paddle index: {cuda_tag} → {paddle_index}')

# ── Gỡ conflict trước ────────────────────────────────────────────────────────
print('\n⏳ Gỡ conflict...')
run('pip uninstall torch torchvision torchaudio paddleocr paddlex easyocr -y -q', 'uninstall conflicts')

# ── Cài PaddlePaddle GPU + PaddleOCR 3.x ────────────────────────────────────
PADDLE_VER = '3.2.1'
PADDLEOCR_VER = '3.3.0'

print(f'\n⏳ Cài paddlepaddle-gpu=={PADDLE_VER} ({cuda_tag})...')
run(f'pip install paddlepaddle-gpu=={PADDLE_VER} -i {paddle_index} -q',
    f'paddlepaddle-gpu=={PADDLE_VER}')

print(f'⏳ Cài paddleocr=={PADDLEOCR_VER}...')
run(f'pip install paddleocr=={PADDLEOCR_VER} -q', f'paddleocr=={PADDLEOCR_VER}')

# ── Patch paddlex để tương thích langchain mới ───────────────────────────────
print('⏳ Patch paddlex...')
run('''python -c "
import pathlib
f = pathlib.Path('/usr/local/lib/python3.12/dist-packages/paddlex/inference/pipelines/components/retriever/base.py')
t = f.read_text()
t = t.replace('from langchain.docstore.document import Document', 'from langchain_core.documents import Document')
t = t.replace('from langchain.text_splitter import RecursiveCharacterTextSplitter', 'from langchain_text_splitters import RecursiveCharacterTextSplitter')
f.write_text(t)
print('patched ok')
"''', 'patch paddlex')

# ── Cài các dependencies ─────────────────────────────────────────────────────
print('⏳ Cài dependencies còn lại...')
run('pip install imagehash yt-dlp opencv-python-headless -q', 'imagehash yt-dlp opencv')
run('pip install langchain-core langchain-text-splitters -q', 'langchain-core')

# ── Kiểm tra ffmpeg ──────────────────────────────────────────────────────────
run('ffmpeg -version 2>&1 | head -1', 'ffmpeg check')

# ── Xác nhận paddle nhận GPU ─────────────────────────────────────────────────
print('\n🔍 Kiểm tra paddle GPU...')
try:
    import paddle
    print(f'   paddle version : {paddle.__version__}')
    print(f'   GPU available  : {paddle.device.cuda.device_count() > 0}')
    print(f'   Device count   : {paddle.device.cuda.device_count()}')
except Exception as e:
    print(f'   ⚠ paddle import lỗi: {e}')

print('\n✅ Cài xong! → Bây giờ RESTART RUNTIME rồi quay lại chạy Cell 1 thêm 1 lần nữa')

OK: nvcc --version

🖥️  CUDA version: 12.8
📦 Paddle index: cu126 → https://www.paddlepaddle.org.cn/packages/stable/cu126/

⏳ Gỡ conflict...
OK: uninstall conflicts

⏳ Cài paddlepaddle-gpu==3.2.1 (cu126)...
OK: paddlepaddle-gpu==3.2.1
⏳ Cài paddleocr==3.3.0...
OK: paddleocr==3.3.0
⏳ Patch paddlex...
OK: patch paddlex
⏳ Cài dependencies còn lại...
OK: imagehash yt-dlp opencv
OK: langchain-core
OK: ffmpeg check

🔍 Kiểm tra paddle GPU...


/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


   paddle version : 3.2.1
   GPU available  : True
   Device count   : 1

✅ Cài xong! → Bây giờ RESTART RUNTIME rồi quay lại chạy Cell 1 thêm 1 lần nữa


In [4]:
# ── CELL 2: Config + Kiểm tra file ──────────────────────────────────────────
import os
from pathlib import Path

# ─── THAY ĐỔI NẾU CẦN ───
DRIVE_BASE    = '/content/drive/MyDrive/PBL7'
META_FILE     = f'{DRIVE_BASE}/metadata.json'
COOKIES_TXT   = f'{DRIVE_BASE}/cookies.txt'
DEBUG_DIR     = f'{DRIVE_BASE}/debug_frames'
TMP_BASE      = '/content/tmp_frames'

BLUR_THRESHOLD  = 100.0
PHASH_THRESHOLD = 5
FPS_EXTRACT     = 0.5  # 1 frame/2s, giảm số frame cần OCR
OCR_CONF_THRESH = 0.1  # Giảm xuống để lấy được nhiều text hơn
JACCARD_THRESH  = 0.6
# ─────────────────────────

os.makedirs(TMP_BASE, exist_ok=True)
os.makedirs(DEBUG_DIR, exist_ok=True)

for f, label in [(META_FILE, 'metadata.json'), (COOKIES_TXT, 'cookies.txt')]:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024
        print(f'✅ {label} ({size:.1f} KB)')
    else:
        print(f'❌ KHNG TIM THẤY: {f}')

import subprocess
r = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader',
                   shell=True, capture_output=True, text=True)
print(f'\n‼️  GPU: {r.stdout.strip() or "Không có GPU (chọn T4 trong Runtime)"}')

✅ metadata.json (3445.1 KB)
✅ cookies.txt (58.4 KB)

‼️  GPU: Tesla T4


In [5]:
# ── CELL 3: Load PaddleOCR ──────────────────────────────────────────────────
import time
import sys
from paddleocr import PaddleOCR

print('⏳ Load PaddleOCR (vi, GPU)...')
t0 = time.time()

# Khởi tạo PaddleOCR 3.x với tham số mới
# use_gpu -> device='gpu'
# use_angle_cls -> use_textline_orientation=True
ocr_engine = PaddleOCR(
    lang='vi',
    device='gpu',
    use_textline_orientation=True
)

print(f'✅ PaddleOCR loaded! ({time.time()-t0:.1f}s)')

Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.


⏳ Load PaddleOCR (vi, GPU)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('UVDoc', None)
Using official model (UVDoc), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/UVDoc`.


...

Creating model: ('PP-OCRv5_server_det', None)
Using official model (PP-OCRv5_server_det), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv5_server_det`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('latin_PP-OCRv5_mobile_rec', None)
Using official model (latin_PP-OCRv5_mobile_rec), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/latin_PP-OCRv5_mobile_rec`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

✅ PaddleOCR loaded! (9.1s)


In [6]:
# ── CELL 4: Random 5 video test ─────────────────────────────────────────────
import json, random

with open(META_FILE, 'r', encoding='utf-8') as f:
    all_records = json.load(f)

# Chỉ lấy video chưa xử lý
unprocessed = [r for r in all_records if r.get('frames') is None]
print(f'📂 Tổng: {len(all_records)} | Chưa xử lý: {len(unprocessed)}')

random.seed(42)
test_videos = random.sample(unprocessed, min(5, len(unprocessed)))

print(f'\n🎲 5 video test (seed=42):')
for i, v in enumerate(test_videos, 1):
    print(f'  {i}. {v["video_id"]} | #{v.get("hashtag_chinh","?")} | {v.get("url","")[:60]}')

📂 Tổng: 5593 | Chưa xử lý: 5447

🎲 5 video test (seed=42):
  1. tk_7634893102984989972 | #xoi | https://www.tiktok.com/@xoimechoux/video/7634893102984989972
  2. tk_7643428928283086087 | #banhbao | https://www.tiktok.com/@the.ham.an.banme/video/7643428928283
  3. tk_7484615230144761096 | #banhbotloc | https://www.tiktok.com/@ducanhhaman/video/748461523014476109
  4. tk_7532812847592656135 | #bunbohue | https://www.tiktok.com/@uneangi/video/7532812847592656135
  5. tk_7509317939498142983 | #bunbohue | https://www.tiktok.com/@kenhcuatienn/video/75093179394981429


In [7]:
import cv2
import shutil
import re
import subprocess
import imagehash
import os
import time
from PIL import Image
from pathlib import Path

# ── OCR với PaddleOCR 3.x ───────────────────────────────────────────────
def ocr_scan(frame_path: str) -> tuple:
    """Trả về (has_text, text_found)"""
    try:
        results = ocr_engine.predict(frame_path)

        if not results or len(results) == 0:
            return False, ''

        lines = []
        for res in results:
            try:
                data = res.json() if hasattr(res, 'json') else res
            except:
                data = res

            texts = []
            scores = []

            if isinstance(data, dict):
                texts = data.get('rec_texts') or data.get('rec_text') or []
                scores = data.get('rec_scores') or data.get('rec_score') or []
            else:
                texts = getattr(data, 'rec_texts', []) or getattr(data, 'rec_text', [])
                scores = getattr(data, 'rec_scores', []) or getattr(data, 'rec_score', [])

            for text, conf in zip(texts, scores):
                if conf > OCR_CONF_THRESH and str(text).strip():
                    lines.append(str(text).strip())

        if not lines:
            return False, ''

        full_text = ' '.join(lines).lower()
        print(f'    [OCR OK] {os.path.basename(frame_path)}: "{full_text[:40]}..."')
        return True, full_text[:200]

    except Exception as e:
        print(f'    ⚠ OCR exception: {e}')
        return False, ''

# ── Filter ────────────────────────────────────────────────
def is_blur(frame_path: str) -> bool:
    img = cv2.imread(frame_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return True
    return cv2.Laplacian(img, cv2.CV_64F).var() < BLUR_THRESHOLD

def is_duplicate(img_path: str, seen_hashes: list) -> bool:
    try:
        h = imagehash.phash(Image.open(img_path))
        for seen in seen_hashes:
            if abs(h - seen) < PHASH_THRESHOLD: return True
        seen_hashes.append(h)
        return False
    except: return True

# ── Text dedup (Jaccard) - Hỗ trợ Tiếng Việt ──────────────────
def normalize_text(text: str) -> set:
    vietnamese_chars = 'a-zàáảãạăắằẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ'
    text = re.sub(f'[{vietnamese_chars} ]', ' ', text.lower())
    return set(text.split())

def jaccard_similarity(a: str, b: str) -> float:
    sa, sb = normalize_text(a), normalize_text(b)
    if not sa or not sb: return 0.0
    return len(sa & sb) / len(sa | sb)

def dedup_texts(ocr_results: list, threshold: float = JACCARD_THRESH) -> list:
    kept = []
    for frame_path, text_found in ocr_results:
        if not any(jaccard_similarity(text_found, kt) >= threshold for _, kt in kept):
            kept.append((frame_path, text_found))
    return kept

# ── Download + Extract (Đã thêm log lỗi chi tiết) ────────────────
def download_video(video_url: str, out_path: str) -> bool:
    try:
        cmd = ['python', '-m', 'yt_dlp', '--cookies', COOKIES_TXT, '--no-playlist', '-f', 'mp4/best', '-o', out_path, '--quiet', '--no-warnings', video_url]
        res = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
        if res.returncode != 0:
            # In lỗi nếu download thất bại để debug
            with print_lock:
                print(f'    ❌ Download error ({video_url.split("/")[-1]}): {res.stderr[:200]}')
            return False
        return os.path.exists(out_path)
    except Exception as e:
        with print_lock:
            print(f'    ❌ Download exception: {e}')
        return False

def extract_frames(video_path: str, out_dir: str) -> list:
    os.makedirs(out_dir, exist_ok=True)
    out_pattern = os.path.join(out_dir, 'frame_%04d.jpg')
    cmd = ['ffmpeg', '-i', video_path, '-vf', f'fps={FPS_EXTRACT}', '-q:v', '2', '-y', out_pattern]
    subprocess.run(cmd, capture_output=True, timeout=120)
    return sorted([str(f) for f in Path(out_dir).glob('frame_*.jpg')])

# ── Process 1 video ─────────────────────────────────────
def process_video_timed(rec: dict) -> dict:
    video_id = rec['video_id']
    hashtag = rec.get('hashtag_chinh', 'unknown')
    tmp_dir = os.path.join(TMP_BASE, video_id)
    os.makedirs(tmp_dir, exist_ok=True)
    video_path = os.path.join(tmp_dir, 'video.mp4')
    debug_dir = Path(DEBUG_DIR) / hashtag / video_id / 'text_filtered'
    debug_dir.mkdir(parents=True, exist_ok=True)

    timer = dict(video_id=video_id, t_download=0, t_extract=0, t_filter=0, t_ocr=0, t_dedup=0, t_total=0, n_raw=0, n_clean=0, n_ocr=0, n_final=0, status='fail', frames_meta=[])
    t_start = time.time()

    if not download_video(rec['url'], video_path): return timer
    timer['t_download'] = time.time() - t_start

    raw_frames = extract_frames(video_path, tmp_dir)
    timer['t_extract'] = time.time() - (t_start + timer['t_download'])
    timer['n_raw'] = len(raw_frames)
    if os.path.exists(video_path): os.remove(video_path)

    seen_hashes = []
    clean_frames = [f for f in raw_frames if not is_blur(f) and not is_duplicate(f, seen_hashes)]
    timer['n_clean'] = len(clean_frames)
    timer['t_filter'] = time.time() - (t_start + timer['t_download'] + timer['t_extract'])

    ocr_results = []
    t_ocr_start = time.time()
    for fp in clean_frames:
        has_text, text = ocr_scan(fp)
        if has_text: ocr_results.append((fp, text))
    timer['t_ocr'] = time.time() - t_ocr_start
    timer['n_ocr'] = len(ocr_results)

    if ocr_results:
        t_dedup_start = time.time()
        filtered = dedup_texts(ocr_results)
        timer['t_dedup'] = time.time() - t_dedup_start
        timer['n_final'] = len(filtered)
        timer['status'] = 'ok'
        for fp, text in filtered:
            shutil.copy2(fp, debug_dir / Path(fp).name)
            timer['frames_meta'].append({'name': Path(fp).name, 'ocr_text': text})

    shutil.rmtree(tmp_dir, ignore_errors=True)
    timer['t_total'] = time.time() - t_start
    return timer

print('✅ Đã cập nhật download_video với log lỗi chi tiết!')

✅ Đã cập nhật download_video với log lỗi chi tiết!


In [ ]:
import time, json, threading, queue, os, shutil, random, subprocess
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# ── CẤU HÌNH NHÓM HASHTAG ──────────────────────────────────────────
TARGET_HASHTAGS = ['bunthitnuong', 'banhxeo', 'miquang']

DOWNLOAD_WORKERS = 1
OCR_WORKERS      = 5
QUEUE_SIZE       = 6
MAX_RETRIES      = 5
# ──────────────────────────────────────────────────────────────────

video_queue   = queue.Queue(maxsize=QUEUE_SIZE)
print_lock    = threading.Lock()
save_lock     = threading.Lock()
meta_lock     = threading.Lock()
final_results = {}

def save_progress(res):
    """Ghi filtered_success.json và metadata.json sau mỗi video OK"""
    with meta_lock:
        # Cập nhật frames vào all_records
        for r in all_records:
            if r['video_id'] == res['video_id']:
                r['frames'] = res['frames_meta']
                break

        # Ghi metadata.json
        with open(META_FILE, 'w', encoding='utf-8') as f:
            json.dump(all_records, f, ensure_ascii=False, indent=2)

        # Merge và ghi filtered_success.json
        entry = {
            'video_id':    res['video_id'],
            'hashtag':     next((r.get('hashtag_chinh', 'unknown') for r in all_records if r['video_id'] == res['video_id']), 'unknown'),
            'url':         next((r.get('url', '') for r in all_records if r['video_id'] == res['video_id']), ''),
            'n_frames':    res.get('n_final', 0),
            'has_text':    res.get('n_final', 0) > 0,
            'frames_meta': res.get('frames_meta', [])
        }
        existing_filtered = []
        if os.path.exists(FILTERED_FILE):
            with open(FILTERED_FILE, 'r', encoding='utf-8') as f:
                existing_filtered = json.load(f)
        if not any(e['video_id'] == res['video_id'] for e in existing_filtered):
            existing_filtered.append(entry)
        with open(FILTERED_FILE, 'w', encoding='utf-8') as f:
            json.dump(existing_filtered, f, ensure_ascii=False, indent=2)

def save_fail(res):
    """Ghi failed_downloads.json ngay khi video fail"""
    with meta_lock:
        existing_failed = []
        if os.path.exists(FAILED_FILE):
            with open(FAILED_FILE, 'r', encoding='utf-8') as f:
                existing_failed = json.load(f)
        if not any(e['video_id'] == res['video_id'] for e in existing_failed):
            existing_failed.append({
                'video_id': res['video_id'],
                'hashtag':  res.get('hashtag', 'unknown'),
                'url':      res.get('url', ''),
                'error':    res.get('error', '')
            })
        with open(FAILED_FILE, 'w', encoding='utf-8') as f:
            json.dump(existing_failed, f, ensure_ascii=False, indent=2)

def internal_process_v2(rec, video_path, tmp_dir, t_dl):
    video_id  = rec['video_id']
    hashtag   = rec.get('hashtag_chinh', 'unknown')
    debug_dir = Path(DEBUG_DIR) / hashtag / video_id / 'text_filtered'
    debug_dir.mkdir(parents=True, exist_ok=True)

    timer = dict(video_id=video_id, t_download=t_dl, t_extract=0,
                 t_filter=0, t_ocr=0, t_total=0,
                 n_raw=0, n_clean=0, n_ocr=0, n_final=0,
                 status='fail', frames_meta=[])
    t_start = time.time()

    try:
        raw_frames = extract_frames(video_path, tmp_dir)
        timer['t_extract'] = time.time() - t_start
        timer['n_raw'] = len(raw_frames)
        if os.path.exists(video_path): os.remove(video_path)

        seen_hashes = []
        clean_frames = [f for f in raw_frames
                        if not is_blur(f) and not is_duplicate(f, seen_hashes)]
        timer['n_clean'] = len(clean_frames)
        timer['t_filter'] = time.time() - t_start - timer['t_extract']

        ocr_res = []
        t_ocr_s = time.time()
        for fp in clean_frames:
            has_text, text = ocr_scan(fp)
            if has_text: ocr_res.append((fp, text))
        timer['t_ocr'] = time.time() - t_ocr_s
        timer['n_ocr'] = len(ocr_res)

        if ocr_res:
            filtered = dedup_texts(ocr_res)
            timer['n_final'] = len(filtered)
            timer['status'] = 'ok'
            for fp, text in filtered:
                shutil.copy2(fp, debug_dir / Path(fp).name)
                timer['frames_meta'].append({'name': Path(fp).name, 'ocr_text': text})
        else:
            timer['status'] = 'ok'
    except Exception as e:
        with print_lock: print(f'  ⚠ Error {video_id}: {e}')
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)
        timer['t_total'] = time.time() - t_start + t_dl
        return timer

def ocr_worker():
    while True:
        item = video_queue.get()
        if item is None: break
        res = internal_process_v2(item['rec'], item['video_path'],
                                   item['tmp_dir'], item['t_dl'])
        final_results[res['video_id']] = res

        # Ghi ngay sau mỗi video ✅
        if res.get('status') == 'ok':
            save_progress(res)
        else:
            save_fail(res)

        with print_lock:
            icon = '✅' if res['status'] == 'ok' else '❌'
            flow = f"{res['n_raw']}→{res['n_clean']}→{res['n_ocr']}→{res['n_final']}"
            print(f"  [{icon}] {res['video_id']} | dl={res['t_download']:.1f}s ocr={res['t_ocr']:.1f}s | {flow}")
        video_queue.task_done()

def downloader_task(rec):
    video_id = rec['video_id']
    url      = rec['url']
    tmp_dir  = os.path.join(TMP_BASE, video_id)
    os.makedirs(tmp_dir, exist_ok=True)
    video_path = os.path.join(tmp_dir, 'video.mp4')
    t_start  = time.time()
    success  = False
    last_err = ''

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            cmd = ['python', '-m', 'yt_dlp',
                   '--cookies', COOKIES_TXT,
                   '--no-playlist', '-f', 'mp4/best',
                   '--retries', '2',
                   '-o', video_path,
                   '--quiet', '--no-warnings', url]
            res = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
            if res.returncode == 0 and os.path.exists(video_path):
                success = True
                break
            last_err = (res.stderr or '').strip().splitlines()[-1] if res.stderr.strip() else 'unknown error'
            with print_lock:
                print(f'  ⚠ Attempt {attempt}/{MAX_RETRIES} fail [{video_id}]: {last_err[:120]}')
            time.sleep(3)
        except Exception as e:
            last_err = str(e)
            time.sleep(5)

    if success:
        dt = time.time() - t_start
        with print_lock: print(f'  [📥 Queued] {video_id} ({dt:.1f}s)')
        video_queue.put({'rec': rec, 'video_path': video_path, 'tmp_dir': tmp_dir, 't_dl': dt})
    else:
        fail_res = {
            'video_id': video_id,
            'hashtag':  rec.get('hashtag_chinh', 'unknown'),
            'url':      url,
            'status':   'dl_fail',
            'error':    last_err[:300]
        }
        final_results[video_id] = fail_res
        save_fail(fail_res)  # Ghi ngay ✅
        with print_lock: print(f'  [❌ Download Fail] {video_id}')

# ── Thực thi ──────────────────────────────────────────────────────
with open(META_FILE, 'r', encoding='utf-8') as f:
    all_records = json.load(f)

FILTERED_FILE = f'{DRIVE_BASE}/filtered_success.json'
FAILED_FILE   = f'{DRIVE_BASE}/failed_downloads.json'

todo = [r for r in all_records
        if r.get('hashtag_chinh') in TARGET_HASHTAGS and r.get('frames') is None]

print(f'🚀 Pipeline 1-DL / {OCR_WORKERS}-OCR | Tổng: {len(todo)} video (không giới hạn batch)')
print(f'🚀 Hashtags: {TARGET_HASHTAGS}')
print('=' * 70)

t_start_batch = time.time()
threads = []
for _ in range(OCR_WORKERS):
    t = threading.Thread(target=ocr_worker, daemon=True)
    t.start(); threads.append(t)

with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
    executor.map(downloader_task, todo)

video_queue.join()
for _ in range(OCR_WORKERS): video_queue.put(None)
for t in threads: t.join()

# ── Báo cáo cuối ──────────────────────────────────────────────────
ok_count       = sum(1 for r in final_results.values() if r.get('status') == 'ok')
has_text_count = sum(1 for r in final_results.values()
                     if r.get('status') == 'ok' and r.get('n_final', 0) > 0)
no_text_count  = sum(1 for r in final_results.values()
                     if r.get('status') == 'ok' and r.get('n_final', 0) == 0)
dl_failed      = [r for r in final_results.values() if r.get('status') == 'dl_fail']
wall           = time.time() - t_start_batch

existing_filtered = []
if os.path.exists(FILTERED_FILE):
    with open(FILTERED_FILE, 'r', encoding='utf-8') as f:
        existing_filtered = json.load(f)

existing_failed = []
if os.path.exists(FAILED_FILE):
    with open(FAILED_FILE, 'r', encoding='utf-8') as f:
        existing_failed = json.load(f)

print('\n' + '=' * 70)
print(f'📊 BẢNG TỔNG KẾT')
print('=' * 70)
print(f'  ✅ Thành công tổng      : {ok_count} video')
print(f'  📝 Có text (OCR đọc được): {has_text_count} video')
print(f'  🔕 Không có text (OK)   : {no_text_count} video  ← bình thường')
print(f'  ❌ Download thất bại    : {len(dl_failed)} video')
print(f'  📄 filtered_success.json : {len(existing_filtered)} video (tổng cộng)')
if dl_failed:
    print(f'  📄 failed_downloads.json : {len(existing_failed)} video (tổng cộng)')
    print()
    print('  Chi tiết thất bại batch này:')
    for e in dl_failed:
        print(f'    [{e["hashtag"]}] {e["video_id"]}')
        print(f'         ↳ {e["error"][:100]}')
print(f'\n  ⏱️  Wall time: {wall:.1f}s  (~{wall/max(ok_count+len(dl_failed),1):.1f}s/video)')
print('=' * 70)

🚀 Pipeline 1-DL / 5-OCR | Tổng: 690 video (không giới hạn batch)
🚀 Hashtags: ['bunthitnuong', 'banhxeo', 'miquang']
  [📥 Queued] tk_7638952452942744839 (4.0s)
  [📥 Queued] tk_7617724020024200456 (4.5s)
  [📥 Queued] tk_7607410944846712086 (9.9s)
  [📥 Queued] tk_7610718376029130005 (10.3s)
  [📥 Queued] tk_7625570983729990930 (12.8s)
    [OCR OK] frame_0001.jpg: "bánh xeo nem nuóng dà thanh ood..."
  [📥 Queued] tk_7612636594474077448 (14.8s)
    [OCR OK] frame_0002.jpg: "phuong u nneg sedrs seses..."
    [OCR OK] frame_0003.jpg: "j o esu..."
    [OCR OK] frame_0004.jpg: "7..."
    [OCR OK] frame_0005.jpg: "94..."
  ⚠ Attempt 1/5 fail [tk_7639225561922948359]: ERROR: [TikTok] 7639225561922948359: Unable to download webpage: HTTP Error 403: Forbidden (caused by <HTTPError 403: Fo
    [OCR OK] frame_0006.jpg: "s oooe s ood southern 0o0 plywood..."
    [OCR OK] frame_0010.jpg: "de dos pon 20010200 bry..."
  ⚠ Attempt 2/5 fail [tk_7639225561922948359]: ERROR: [TikTok] 7639225561922948359: Unab

resume


In [8]:
import time, json, threading, queue, os, shutil, random, subprocess
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# ── CẤU HÌNH NHÓM HASHTAG ──────────────────────────────────────────
TARGET_HASHTAGS = ['bunthitnuong', 'banhxeo', 'miquang']

DOWNLOAD_WORKERS = 1
OCR_WORKERS      = 4
QUEUE_SIZE       = 6
MAX_RETRIES      = 5
# ──────────────────────────────────────────────────────────────────

video_queue   = queue.Queue(maxsize=QUEUE_SIZE)
print_lock    = threading.Lock()
save_lock     = threading.Lock()
meta_lock     = threading.Lock()
final_results = {}

def save_progress(res):
    with meta_lock:
        for r in all_records:
            if r['video_id'] == res['video_id']:
                r['frames'] = res['frames_meta']
                break
        with open(META_FILE, 'w', encoding='utf-8') as f:
            json.dump(all_records, f, ensure_ascii=False, indent=2)

        entry = {
            'video_id':    res['video_id'],
            'hashtag':     next((r.get('hashtag_chinh', 'unknown') for r in all_records if r['video_id'] == res['video_id']), 'unknown'),
            'url':         next((r.get('url', '') for r in all_records if r['video_id'] == res['video_id']), ''),
            'n_frames':    res.get('n_final', 0),
            'has_text':    res.get('n_final', 0) > 0,
            'frames_meta': res.get('frames_meta', [])
        }
        existing_filtered = []
        if os.path.exists(FILTERED_FILE):
            with open(FILTERED_FILE, 'r', encoding='utf-8') as f:
                existing_filtered = json.load(f)
        if not any(e['video_id'] == res['video_id'] for e in existing_filtered):
            existing_filtered.append(entry)
        with open(FILTERED_FILE, 'w', encoding='utf-8') as f:
            json.dump(existing_filtered, f, ensure_ascii=False, indent=2)

def save_fail(res):
    with meta_lock:
        existing_failed = []
        if os.path.exists(FAILED_FILE):
            with open(FAILED_FILE, 'r', encoding='utf-8') as f:
                existing_failed = json.load(f)
        if not any(e['video_id'] == res['video_id'] for e in existing_failed):
            existing_failed.append({
                'video_id': res['video_id'],
                'hashtag':  res.get('hashtag', 'unknown'),
                'url':      res.get('url', ''),
                'error':    res.get('error', '')
            })
        with open(FAILED_FILE, 'w', encoding='utf-8') as f:
            json.dump(existing_failed, f, ensure_ascii=False, indent=2)

def internal_process_v2(rec, video_path, tmp_dir, t_dl):
    video_id  = rec['video_id']
    hashtag   = rec.get('hashtag_chinh', 'unknown')
    debug_dir = Path(DEBUG_DIR) / hashtag / video_id / 'text_filtered'
    debug_dir.mkdir(parents=True, exist_ok=True)

    timer = dict(video_id=video_id, t_download=t_dl, t_extract=0,
                 t_filter=0, t_ocr=0, t_total=0,
                 n_raw=0, n_clean=0, n_ocr=0, n_final=0,
                 status='fail', frames_meta=[])
    t_start = time.time()

    try:
        raw_frames = extract_frames(video_path, tmp_dir)
        timer['t_extract'] = time.time() - t_start
        timer['n_raw'] = len(raw_frames)
        if os.path.exists(video_path): os.remove(video_path)

        seen_hashes = []
        clean_frames = [f for f in raw_frames
                        if not is_blur(f) and not is_duplicate(f, seen_hashes)]
        timer['n_clean'] = len(clean_frames)
        timer['t_filter'] = time.time() - t_start - timer['t_extract']

        ocr_res = []
        t_ocr_s = time.time()
        for fp in clean_frames:
            has_text, text = ocr_scan(fp)
            if has_text: ocr_res.append((fp, text))
        timer['t_ocr'] = time.time() - t_ocr_s
        timer['n_ocr'] = len(ocr_res)

        if ocr_res:
            filtered = dedup_texts(ocr_res)
            timer['n_final'] = len(filtered)
            timer['status'] = 'ok'
            for fp, text in filtered:
                shutil.copy2(fp, debug_dir / Path(fp).name)
                timer['frames_meta'].append({'name': Path(fp).name, 'ocr_text': text})
        else:
            timer['status'] = 'ok'
    except Exception as e:
        with print_lock: print(f'  ⚠ Error {video_id}: {e}')
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)
        timer['t_total'] = time.time() - t_start + t_dl
        return timer

def ocr_worker():
    while True:
        item = video_queue.get()
        if item is None: break
        res = internal_process_v2(item['rec'], item['video_path'],
                                   item['tmp_dir'], item['t_dl'])
        final_results[res['video_id']] = res

        if res.get('status') == 'ok':
            save_progress(res)
        else:
            save_fail(res)

        with print_lock:
            icon = '✅' if res['status'] == 'ok' else '❌'
            flow = f"{res['n_raw']}→{res['n_clean']}→{res['n_ocr']}→{res['n_final']}"
            print(f"  [{icon}] {res['video_id']} | dl={res['t_download']:.1f}s ocr={res['t_ocr']:.1f}s | {flow}")
        video_queue.task_done()

def downloader_task(rec):
    video_id = rec['video_id']
    url      = rec['url']
    tmp_dir  = os.path.join(TMP_BASE, video_id)
    os.makedirs(tmp_dir, exist_ok=True)
    video_path = os.path.join(tmp_dir, 'video.mp4')
    t_start  = time.time()
    success  = False
    last_err = ''

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            cmd = ['python', '-m', 'yt_dlp',
                   '--cookies', COOKIES_TXT,
                   '--no-playlist', '-f', 'mp4/best',
                   '--retries', '2',
                   '-o', video_path,
                   '--quiet', '--no-warnings', url]
            res = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
            if res.returncode == 0 and os.path.exists(video_path):
                success = True
                break
            last_err = (res.stderr or '').strip().splitlines()[-1] if res.stderr.strip() else 'unknown error'
            with print_lock:
                print(f'  ⚠ Attempt {attempt}/{MAX_RETRIES} fail [{video_id}]: {last_err[:120]}')
            time.sleep(3)
        except Exception as e:
            last_err = str(e)
            time.sleep(5)

    if success:
        dt = time.time() - t_start
        with print_lock: print(f'  [📥 Queued] {video_id} ({dt:.1f}s)')
        video_queue.put({'rec': rec, 'video_path': video_path, 'tmp_dir': tmp_dir, 't_dl': dt})
    else:
        fail_res = {
            'video_id': video_id,
            'hashtag':  rec.get('hashtag_chinh', 'unknown'),
            'url':      url,
            'status':   'dl_fail',
            'error':    last_err[:300]
        }
        final_results[video_id] = fail_res
        save_fail(fail_res)
        with print_lock: print(f'  [❌ Download Fail] {video_id}')

# ── Thực thi ──────────────────────────────────────────────────────
with open(META_FILE, 'r', encoding='utf-8') as f:
    all_records = json.load(f)

FILTERED_FILE = f'{DRIVE_BASE}/filtered_success.json'
FAILED_FILE   = f'{DRIVE_BASE}/failed_downloads.json'

# Skip video đã xử lý dựa vào filtered_success.json ✅
processed_ids = set()
if os.path.exists(FILTERED_FILE):
    with open(FILTERED_FILE, 'r', encoding='utf-8') as f:
        processed_ids = {e['video_id'] for e in json.load(f)}

todo = [r for r in all_records
        if r.get('hashtag_chinh') in TARGET_HASHTAGS
        and r['video_id'] not in processed_ids]

print(f'🚀 Pipeline 1-DL / {OCR_WORKERS}-OCR | Tổng: {len(todo)} video (không giới hạn batch)')
print(f'🚀 Hashtags: {TARGET_HASHTAGS}')
print(f'⏭️  Đã xử lý trước đó: {len(processed_ids)} video → bỏ qua')
print('=' * 70)

t_start_batch = time.time()
threads = []
for _ in range(OCR_WORKERS):
    t = threading.Thread(target=ocr_worker, daemon=True)
    t.start(); threads.append(t)

with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
    executor.map(downloader_task, todo)

video_queue.join()
for _ in range(OCR_WORKERS): video_queue.put(None)
for t in threads: t.join()

# ── Báo cáo cuối ──────────────────────────────────────────────────
ok_count       = sum(1 for r in final_results.values() if r.get('status') == 'ok')
has_text_count = sum(1 for r in final_results.values()
                     if r.get('status') == 'ok' and r.get('n_final', 0) > 0)
no_text_count  = sum(1 for r in final_results.values()
                     if r.get('status') == 'ok' and r.get('n_final', 0) == 0)
dl_failed      = [r for r in final_results.values() if r.get('status') == 'dl_fail']
wall           = time.time() - t_start_batch

existing_filtered = []
if os.path.exists(FILTERED_FILE):
    with open(FILTERED_FILE, 'r', encoding='utf-8') as f:
        existing_filtered = json.load(f)

existing_failed = []
if os.path.exists(FAILED_FILE):
    with open(FAILED_FILE, 'r', encoding='utf-8') as f:
        existing_failed = json.load(f)

print('\n' + '=' * 70)
print(f'📊 BẢNG TỔNG KẾT')
print('=' * 70)
print(f'  ✅ Thành công tổng      : {ok_count} video')
print(f'  📝 Có text (OCR đọc được): {has_text_count} video')
print(f'  🔕 Không có text (OK)   : {no_text_count} video  ← bình thường')
print(f'  ❌ Download thất bại    : {len(dl_failed)} video')
print(f'  📄 filtered_success.json : {len(existing_filtered)} video (tổng cộng)')
if dl_failed:
    print(f'  📄 failed_downloads.json : {len(existing_failed)} video (tổng cộng)')
    print()
    print('  Chi tiết thất bại batch này:')
    for e in dl_failed:
        print(f'    [{e["hashtag"]}] {e["video_id"]}')
        print(f'         ↳ {e["error"][:100]}')
print(f'\n  ⏱️  Wall time: {wall:.1f}s  (~{wall/max(ok_count+len(dl_failed),1):.1f}s/video)')
print('=' * 70)

Streaming output truncated to the last 5000 lines.
    [OCR OK] frame_0015.jpg: "ri..."
  [📥 Queued] tk_7624943682340375826 (3.9s)
  ⚠ Attempt 1/5 fail [tk_7556578539974216978]: ERROR: [TikTok] 7556578539974216978: Unable to extract universal data for rehydration; please report this issue on  http
    [OCR OK] frame_0024.jpg: "sng sánh hi 05.76y 20k nung bún thit hng..."
  [✅] tk_7499833569158663432 | dl=5.8s ocr=15.5s | 24→11→8→6
  ⚠ Attempt 2/5 fail [tk_7556578539974216978]: ERROR: [TikTok] 7556578539974216978: Unable to download webpage: HTTP Error 403: Forbidden (caused by <HTTPError 403: Fo
  ⚠ Attempt 3/5 fail [tk_7556578539974216978]: ERROR: [TikTok] 7556578539974216978: Unable to download webpage: HTTP Error 403: Forbidden (caused by <HTTPError 403: Fo
    [OCR OK] frame_0001.jpg: "series gân uan ngon cdc truòn..."
    [OCR OK] frame_0002.jpg: "ngonré 46k khu đóng đa gân dk các trưòng..."
    [OCR OK] frame_0003.jpg: "thuli ngân hàng công đon..."
    [OCR OK] frame_0004.jpg: "1

check d fail

In [ ]:
import time, json, threading, queue, os, shutil, random, subprocess
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# ── CẤU HÌNH NHÓM HASHTAG ──────────────────────────────────────────
TARGET_HASHTAGS = ['bunbohue', 'pho', 'banhkhot', 'caolau']

DOWNLOAD_WORKERS = 1
OCR_WORKERS      = 5
QUEUE_SIZE       = 6
MAX_RETRIES      = 5
# ──────────────────────────────────────────────────────────────────

video_queue   = queue.Queue(maxsize=QUEUE_SIZE)
print_lock    = threading.Lock()
save_lock     = threading.Lock()
meta_lock     = threading.Lock()
final_results = {}

def save_progress(res):
    with meta_lock:
        for r in all_records:
            if r['video_id'] == res['video_id']:
                r['frames'] = res['frames_meta']
                break
        with open(META_FILE, 'w', encoding='utf-8') as f:
            json.dump(all_records, f, ensure_ascii=False, indent=2)

        entry = {
            'video_id':    res['video_id'],
            'hashtag':     next((r.get('hashtag_chinh', 'unknown') for r in all_records if r['video_id'] == res['video_id']), 'unknown'),
            'url':         next((r.get('url', '') for r in all_records if r['video_id'] == res['video_id']), ''),
            'n_frames':    res.get('n_final', 0),
            'has_text':    res.get('n_final', 0) > 0,
            'frames_meta': res.get('frames_meta', [])
        }
        existing_filtered = []
        if os.path.exists(FILTERED_FILE):
            with open(FILTERED_FILE, 'r', encoding='utf-8') as f:
                existing_filtered = json.load(f)
        if not any(e['video_id'] == res['video_id'] for e in existing_filtered):
            existing_filtered.append(entry)
        with open(FILTERED_FILE, 'w', encoding='utf-8') as f:
            json.dump(existing_filtered, f, ensure_ascii=False, indent=2)

def save_fail(res):
    with meta_lock:
        existing_failed = []
        if os.path.exists(FAILED_FILE):
            with open(FAILED_FILE, 'r', encoding='utf-8') as f:
                existing_failed = json.load(f)
        if not any(e['video_id'] == res['video_id'] for e in existing_failed):
            existing_failed.append({
                'video_id': res['video_id'],
                'hashtag':  res.get('hashtag', 'unknown'),
                'url':      res.get('url', ''),
                'error':    res.get('error', '')
            })
        with open(FAILED_FILE, 'w', encoding='utf-8') as f:
            json.dump(existing_failed, f, ensure_ascii=False, indent=2)

def internal_process_v2(rec, video_path, tmp_dir, t_dl):
    video_id  = rec['video_id']
    hashtag   = rec.get('hashtag_chinh', 'unknown')
    debug_dir = Path(DEBUG_DIR) / hashtag / video_id / 'text_filtered'
    debug_dir.mkdir(parents=True, exist_ok=True)

    timer = dict(video_id=video_id, t_download=t_dl, t_extract=0,
                 t_filter=0, t_ocr=0, t_total=0,
                 n_raw=0, n_clean=0, n_ocr=0, n_final=0,
                 status='fail', frames_meta=[])
    t_start = time.time()

    try:
        raw_frames = extract_frames(video_path, tmp_dir)
        timer['t_extract'] = time.time() - t_start
        timer['n_raw'] = len(raw_frames)
        if os.path.exists(video_path): os.remove(video_path)

        seen_hashes = []
        clean_frames = [f for f in raw_frames
                        if not is_blur(f) and not is_duplicate(f, seen_hashes)]
        timer['n_clean'] = len(clean_frames)
        timer['t_filter'] = time.time() - t_start - timer['t_extract']

        ocr_res = []
        t_ocr_s = time.time()
        for fp in clean_frames:
            has_text, text = ocr_scan(fp)
            if has_text: ocr_res.append((fp, text))
        timer['t_ocr'] = time.time() - t_ocr_s
        timer['n_ocr'] = len(ocr_res)

        if ocr_res:
            filtered = dedup_texts(ocr_res)
            timer['n_final'] = len(filtered)
            timer['status'] = 'ok'
            for fp, text in filtered:
                shutil.copy2(fp, debug_dir / Path(fp).name)
                timer['frames_meta'].append({'name': Path(fp).name, 'ocr_text': text})
        else:
            timer['status'] = 'ok'
    except Exception as e:
        with print_lock: print(f'  ⚠ Error {video_id}: {e}')
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)
        timer['t_total'] = time.time() - t_start + t_dl
        return timer

def ocr_worker():
    while True:
        item = video_queue.get()
        if item is None: break
        res = internal_process_v2(item['rec'], item['video_path'],
                                   item['tmp_dir'], item['t_dl'])
        final_results[res['video_id']] = res

        if res.get('status') == 'ok':
            save_progress(res)
        else:
            save_fail(res)

        with print_lock:
            icon = '✅' if res['status'] == 'ok' else '❌'
            flow = f"{res['n_raw']}→{res['n_clean']}→{res['n_ocr']}→{res['n_final']}"
            print(f"  [{icon}] {res['video_id']} | dl={res['t_download']:.1f}s ocr={res['t_ocr']:.1f}s | {flow}")
        video_queue.task_done()

def downloader_task(rec):
    video_id = rec['video_id']
    url      = rec['url']
    tmp_dir  = os.path.join(TMP_BASE, video_id)
    os.makedirs(tmp_dir, exist_ok=True)
    video_path = os.path.join(tmp_dir, 'video.mp4')
    t_start  = time.time()
    success  = False
    last_err = ''

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            cmd = ['python', '-m', 'yt_dlp',
                   '--cookies', COOKIES_TXT,
                   '--no-playlist', '-f', 'mp4/best',
                   '--retries', '2',
                   '-o', video_path,
                   '--quiet', '--no-warnings', url]
            res = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
            if res.returncode == 0 and os.path.exists(video_path):
                success = True
                break
            last_err = (res.stderr or '').strip().splitlines()[-1] if res.stderr.strip() else 'unknown error'
            with print_lock:
                print(f'  ⚠ Attempt {attempt}/{MAX_RETRIES} fail [{video_id}]: {last_err[:120]}')
            time.sleep(3)
        except Exception as e:
            last_err = str(e)
            time.sleep(5)

    if success:
        dt = time.time() - t_start
        with print_lock: print(f'  [📥 Queued] {video_id} ({dt:.1f}s)')
        video_queue.put({'rec': rec, 'video_path': video_path, 'tmp_dir': tmp_dir, 't_dl': dt})
    else:
        fail_res = {
            'video_id': video_id,
            'hashtag':  rec.get('hashtag_chinh', 'unknown'),
            'url':      url,
            'status':   'dl_fail',
            'error':    last_err[:300]
        }
        final_results[video_id] = fail_res
        save_fail(fail_res)
        with print_lock: print(f'  [❌ Download Fail] {video_id}')

# ── Thực thi ──────────────────────────────────────────────────────
with open(META_FILE, 'r', encoding='utf-8') as f:
    all_records = json.load(f)

FILTERED_FILE = f'{DRIVE_BASE}/filtered_success.json'
FAILED_FILE   = f'{DRIVE_BASE}/failed_downloads.json'

# Đọc danh sách đã thành công ✅
processed_ids = set()
if os.path.exists(FILTERED_FILE):
    with open(FILTERED_FILE, 'r', encoding='utf-8') as f:
        processed_ids = {e['video_id'] for e in json.load(f)}

# Đọc danh sách video fail thuộc Colab này, bỏ qua cái đã success ✅
with open(FAILED_FILE, 'r', encoding='utf-8') as f:
    failed_records = json.load(f)

retry_ids = {e['video_id'] for e in failed_records
             if e.get('hashtag') in TARGET_HASHTAGS
             and e['video_id'] not in processed_ids}

todo = [r for r in all_records if r['video_id'] in retry_ids]

if not todo:
    print('✅ Không có video nào cần retry')
else:
    print(f'🔁 Retry {len(todo)} video fail | Hashtags: {TARGET_HASHTAGS}')
    print('=' * 70)

    t_start_batch = time.time()
    threads = []
    for _ in range(OCR_WORKERS):
        t = threading.Thread(target=ocr_worker, daemon=True)
        t.start(); threads.append(t)

    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
        executor.map(downloader_task, todo)

    video_queue.join()
    for _ in range(OCR_WORKERS): video_queue.put(None)
    for t in threads: t.join()

    # ── Báo cáo cuối ──────────────────────────────────────────────
    ok_count       = sum(1 for r in final_results.values() if r.get('status') == 'ok')
    has_text_count = sum(1 for r in final_results.values()
                         if r.get('status') == 'ok' and r.get('n_final', 0) > 0)
    no_text_count  = sum(1 for r in final_results.values()
                         if r.get('status') == 'ok' and r.get('n_final', 0) == 0)
    dl_failed      = [r for r in final_results.values() if r.get('status') == 'dl_fail']
    wall           = time.time() - t_start_batch

    existing_filtered = []
    if os.path.exists(FILTERED_FILE):
        with open(FILTERED_FILE, 'r', encoding='utf-8') as f:
            existing_filtered = json.load(f)

    existing_failed = []
    if os.path.exists(FAILED_FILE):
        with open(FAILED_FILE, 'r', encoding='utf-8') as f:
            existing_failed = json.load(f)

    print('\n' + '=' * 70)
    print(f'📊 BẢNG TỔNG KẾT RETRY')
    print('=' * 70)
    print(f'  ✅ Thành công tổng      : {ok_count} video')
    print(f'  📝 Có text (OCR đọc được): {has_text_count} video')
    print(f'  🔕 Không có text (OK)   : {no_text_count} video  ← bình thường')
    print(f'  ❌ Vẫn còn fail         : {len(dl_failed)} video')
    print(f'  📄 filtered_success.json : {len(existing_filtered)} video (tổng cộng)')
    if dl_failed:
        print(f'  📄 failed_downloads.json : {len(existing_failed)} video (tổng cộng)')
        print()
        print('  Chi tiết vẫn còn fail:')
        for e in dl_failed:
            print(f'    [{e["hashtag"]}] {e["video_id"]}')
            print(f'         ↳ {e["error"][:100]}')
    print(f'\n  ⏱️  Wall time: {wall:.1f}s  (~{wall/max(ok_count+len(dl_failed),1):.1f}s/video)')
    print('=' * 70)